<a href="https://colab.research.google.com/github/Saikadam123/ADM-Project/blob/main/CBP_Multimodal_Baseline_Optimized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal CXR Abnormality Classification
## CheXpert DenseNet121 + BioClinicalBERT + Residual Compact Bilinear Pooling



## 1. Environment Setup


In [ ]:
# Install the required packages.
!pip install -q transformers accelerate scikit-learn huggingface_hub safetensors --upgrade


In [ ]:
# Import required libraries.
import os
import re
import json
import math
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from tqdm.auto import tqdm

from torchvision import models, transforms
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Google Drive Access


In [ ]:
# Mount Google Drive to access the dataset and save outputs.
from google.colab import drive
drive.mount('/content/drive')


## 3. Experiment Configuration


In [ ]:
# Define dataset paths, model settings and training hyperparameters.

DATASET_DIR = "/content/drive/MyDrive/Dataset"

IMAGE_FOLDER = os.path.join(DATASET_DIR, "images_normalized")
REPORTS_PATH = os.path.join(DATASET_DIR, "indiana_reports.csv")

TRAIN_CSV = os.path.join(DATASET_DIR, "train.csv")
VAL_CSV   = os.path.join(DATASET_DIR, "validation.csv")
TEST_CSV  = os.path.join(DATASET_DIR, "test.csv")

MODEL_FOLDER = os.path.join(DATASET_DIR, "trained_model_cbp_e2e_v2")
os.makedirs(MODEL_FOLDER, exist_ok=True)

BEST_MODEL_PATH = os.path.join(MODEL_FOLDER, "best_cbp_e2e_v2_model.pth")
CHEXPERT_REPO_ID = "itsomk/chexpert-densenet121"
CHEXPERT_WEIGHT_FILENAMES = ("pytorch_model.safetensors", "chexpert_pytorch.safetensors")
TEXT_MODEL_NAME  = "emilyalsentzer/Bio_ClinicalBERT"

MAX_TEXT_LEN = 256
IMAGE_SIZE = 224
DENSENET_IMAGE_MEAN = [0.485, 0.456, 0.406]
DENSENET_IMAGE_STD = [0.229, 0.224, 0.225]

USE_TEXT_ATTENTION_POOLING = True
USE_IMAGE_ATTENTION_POOLING = True

DENSENET_FEATURE_DIM = 1024
TEXT_DIM   = 768
IMAGE_DIM  = 768
CBP_PROJ_DIM = 192
CBP_CONV_FILTERS = 4
FUSION_DIM = 256

NUM_UNFROZEN_DENSENET_BLOCKS = 2
NUM_UNFROZEN_BERT_LAYERS = 2
PHASE1_EPOCHS = 2
PHASE2_MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5

WARMUP_RATIO_PHASE2 = 0.1
BACKBONE_LR = 2e-5
HEAD_LR     = 1e-3
WEIGHT_DECAY = 1e-4

PHYSICAL_BATCH_SIZE = 4
EFFECTIVE_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = max(1, EFFECTIVE_BATCH_SIZE // PHYSICAL_BATCH_SIZE)

MAX_GRAD_NORM = 1.0
NUM_WORKERS = 2
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print(f"Physical batch size: {PHYSICAL_BATCH_SIZE} | "
      f"Grad accumulation steps: {GRAD_ACCUM_STEPS} | "
      f"Effective batch size: {PHYSICAL_BATCH_SIZE * GRAD_ACCUM_STEPS}")


In [ ]:
# Set random seeds and configure the computation device.
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


## 4. Data Loading and Class Distribution


In [ ]:
# Load the train, validation and test splits and inspect class balance.
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(df)} samples")
    print(df["binary_label"].value_counts(normalize=True).sort_index(), "\n")

pos_frac = train_df["binary_label"].mean()
print(f"Training positive-class fraction: {pos_frac:.4f}")


## 5. Dataset, Preprocessing, and DataLoaders

In [ ]:
# Define the paired CXR–report dataset used for multimodal learning.
class CXRMultimodalDataset(Dataset):
    def __init__(self, df, image_folder, image_processor, tokenizer, max_text_len=256):
        self.df = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.max_text_len = max_text_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = os.path.join(self.image_folder, row["filename"])
        with Image.open(image_path) as img:
            image = img.convert("RGB")

        pixel_values = self.image_processor(
            images=image,
            return_tensors="pt"
        )["pixel_values"].squeeze(0)

        findings = str(row["findings"])
        encoded = self.tokenizer(
            findings,
            padding="max_length",
            truncation=True,
            max_length=self.max_text_len,
            return_tensors="pt"
        )
        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)

        label = torch.tensor(row["binary_label"], dtype=torch.float32)

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": label,
            "uid": row["uid"],
        }


In [ ]:
# Define batch collation for image, text, label and UID tensors.
def collate_fn(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "label": torch.stack([b["label"] for b in batch]),
        "uid": [b["uid"] for b in batch],
    }


In [ ]:
# Define DenseNet compatible image preprocessing and initialize data loaders.
class DenseNetImageProcessor:
    """DenseNet preprocessing with the same dataset-facing interface used by the baseline."""
    def __init__(self, image_size=IMAGE_SIZE):
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=DENSENET_IMAGE_MEAN, std=DENSENET_IMAGE_STD),
        ])

    def __call__(self, images, return_tensors="pt"):
        if return_tensors != "pt":
            raise ValueError("DenseNetImageProcessor supports return_tensors='pt' only.")
        x = self.transform(images.convert("RGB"))
        return {"pixel_values": x.unsqueeze(0)}


image_processor = DenseNetImageProcessor()
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

train_dataset = CXRMultimodalDataset(train_df, IMAGE_FOLDER, image_processor, tokenizer, MAX_TEXT_LEN)
val_dataset   = CXRMultimodalDataset(val_df,   IMAGE_FOLDER, image_processor, tokenizer, MAX_TEXT_LEN)
test_dataset  = CXRMultimodalDataset(test_df,  IMAGE_FOLDER, image_processor, tokenizer, MAX_TEXT_LEN)

train_loader = DataLoader(
    train_dataset, batch_size=PHYSICAL_BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_dataset, batch_size=PHYSICAL_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_dataset, batch_size=PHYSICAL_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn,
)

print("DenseNet image preprocessing: RGB -> 224x224 -> ToTensor -> checkpoint normalization")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))


## 6. Attention Pooling


In [ ]:
# Define learned attention pooling for text tokens and image spatial features.
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, hidden_states, attention_mask=None):

        scores = self.attn(hidden_states).squeeze(-1)

        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask == 0, float("-inf"))

        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        pooled = (hidden_states * weights).sum(dim=1)

        return pooled, weights.squeeze(-1)


## 7. Residual Compact Bilinear Pooling


In [ ]:
# Define residual compact bilinear pooling for multimodal feature fusion.
class CompactBilinearPoolingCNN(nn.Module):
    def __init__(
        self,
        text_dim=768,
        image_dim=768,
        proj_dim=192,
        conv_filters=4,
        fusion_dim=256,
        dropout=0.3,
    ):
        super().__init__()

        self.proj_dim = proj_dim

        self.text_proj = nn.Linear(text_dim, proj_dim)
        self.image_proj = nn.Linear(image_dim, proj_dim)
        self.proj_dropout = nn.Dropout(dropout)

        self.conv_block = nn.Sequential(
            nn.Conv2d(1, conv_filters, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )

        pooled_dim = proj_dim // 2
        conv_out_dim = conv_filters * pooled_dim * pooled_dim


        fusion_in_dim = conv_out_dim + proj_dim + proj_dim

        self.fusion_proj = nn.Sequential(
            nn.Linear(fusion_in_dim, fusion_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )

    def forward(self, text_features, image_features):
        t = self.proj_dropout(self.text_proj(text_features))
        v = self.proj_dropout(self.image_proj(image_features))

        bilinear_map = torch.bmm(v.unsqueeze(2), t.unsqueeze(1))
        bilinear_map = bilinear_map.unsqueeze(1)

        compressed = self.conv_block(bilinear_map)
        compressed = compressed.flatten(start_dim=1)

        fusion_input = torch.cat([compressed, t, v], dim=1)
        fused = self.fusion_proj(fusion_input)

        return fused


## 8. Multimodal Model Architecture


In [ ]:
# Load verified CheXpert DenseNet121 weights and define the multimodal classifier.
def _download_chexpert_weights():
    """Download the validated CheXpert DenseNet121 safetensors checkpoint."""
    last_error = None
    for filename in CHEXPERT_WEIGHT_FILENAMES:
        try:
            path = hf_hub_download(repo_id=CHEXPERT_REPO_ID, filename=filename)
            print(f"CheXpert checkpoint source: {CHEXPERT_REPO_ID}/{filename}")
            print("CheXpert checkpoint path  :", path)
            return path
        except Exception as exc:
            last_error = exc
    raise RuntimeError(
        f"Could not download a supported checkpoint file from {CHEXPERT_REPO_ID}: {last_error}"
    )


def build_verified_chexpert_densenet121():
    """Load torchvision DenseNet121 and require complete CheXpert features.* coverage."""
    checkpoint_path = _download_chexpert_weights()
    raw_state = load_file(checkpoint_path)

    had_wrapper_prefix = any(k.startswith("densenet.") for k in raw_state)
    if had_wrapper_prefix:
        state_dict = {
            k[len("densenet."):] if k.startswith("densenet.") else k: v
            for k, v in raw_state.items()
        }
    else:
        state_dict = dict(raw_state)


    checkpoint_feature_state = {k: v for k, v in state_dict.items() if k.startswith("features.")}
    checkpoint_classifier_keys = [k for k in state_dict if k.startswith("classifier.")]
    if not checkpoint_feature_state:
        raise RuntimeError("No features.* tensors were found in the CheXpert checkpoint.")

    backbone = models.densenet121(weights=None)
    fresh = models.densenet121(weights=None)
    load_report = backbone.load_state_dict(checkpoint_feature_state, strict=False)

    backbone_missing = [k for k in load_report.missing_keys if k.startswith("features.")]
    unexpected_features = [k for k in load_report.unexpected_keys if k.startswith("features.")]
    model_feature_keys = {k for k in backbone.state_dict() if k.startswith("features.")}
    loaded_feature_keys = sorted(model_feature_keys.intersection(checkpoint_feature_state.keys()))

    if backbone_missing:
        raise RuntimeError(
            f"CheXpert DenseNet backbone failed to load: {len(backbone_missing)} features.* keys are missing. "
            f"First missing keys: {backbone_missing[:10]}"
        )
    if unexpected_features:
        raise RuntimeError(f"Unexpected CheXpert features.* keys: {unexpected_features[:10]}")
    if len(loaded_feature_keys) != len(model_feature_keys):
        missing_by_set = sorted(model_feature_keys.difference(checkpoint_feature_state.keys()))
        raise RuntimeError(
            f"CheXpert DenseNet feature-key coverage incomplete: "
            f"{len(loaded_feature_keys)}/{len(model_feature_keys)} loaded. "
            f"First absent keys: {missing_by_set[:10]}"
        )

    probe_key = "features.conv0.weight"
    differs_from_fresh = not torch.allclose(
        backbone.state_dict()[probe_key].detach().cpu(),
        fresh.state_dict()[probe_key].detach().cpu(),
    )
    del fresh
    if not differs_from_fresh:
        raise RuntimeError("Loaded DenseNet features.conv0.weight matches a fresh initialization.")


    backbone.classifier = nn.Identity()

    report = {
        "repo_id": CHEXPERT_REPO_ID,
        "checkpoint_path": checkpoint_path,
        "wrapper_prefix_removed": bool(had_wrapper_prefix),
        "checkpoint_feature_keys": len(checkpoint_feature_state),
        "loaded_feature_keys": len(loaded_feature_keys),
        "missing_backbone_feature_keys": len(backbone_missing),
        "unexpected_feature_keys": len(unexpected_features),
        "ignored_classifier_keys": len(checkpoint_classifier_keys),
        "differs_from_fresh_init": bool(differs_from_fresh),
    }

    print("\nSUCCESS: CheXpert-pretrained DenseNet121 backbone loaded.")
    print("Removed 'densenet.' prefix              :", report["wrapper_prefix_removed"])
    print("DenseNet features.* keys in checkpoint :", report["checkpoint_feature_keys"])
    print("Loaded DenseNet features.* keys        :", report["loaded_feature_keys"])
    print("Missing backbone features.* keys       :", report["missing_backbone_feature_keys"])
    print("Unexpected DenseNet feature keys       :", report["unexpected_feature_keys"])
    print("Ignored original classifier keys       :", report["ignored_classifier_keys"])
    print("Different from fresh DenseNet121       :", report["differs_from_fresh_init"])
    return backbone, report


class MultimodalCBPClassifier(nn.Module):
    def __init__(
        self,
        text_model_name=TEXT_MODEL_NAME,
        cbp_proj_dim=CBP_PROJ_DIM,
        cbp_conv_filters=CBP_CONV_FILTERS,
        fusion_dim=FUSION_DIM,
        classifier_dropout=0.3,
        use_text_attention_pooling=USE_TEXT_ATTENTION_POOLING,
        use_image_attention_pooling=USE_IMAGE_ATTENTION_POOLING,
    ):
        super().__init__()

        self.image_encoder, self.chexpert_load_report = build_verified_chexpert_densenet121()
        self.text_encoder = AutoModel.from_pretrained(text_model_name)

        self.use_text_attention_pooling = use_text_attention_pooling
        self.use_image_attention_pooling = use_image_attention_pooling

        if self.use_text_attention_pooling:
            self.text_attn_pool = AttentionPooling(self.text_encoder.config.hidden_size)

        if self.use_image_attention_pooling:
            self.image_attn_pool = AttentionPooling(DENSENET_FEATURE_DIM)


        self.image_projection = nn.Sequential(
            nn.LayerNorm(DENSENET_FEATURE_DIM),
            nn.Linear(DENSENET_FEATURE_DIM, IMAGE_DIM),
        )

        self.cbp = CompactBilinearPoolingCNN(
            text_dim=TEXT_DIM,
            image_dim=IMAGE_DIM,
            proj_dim=cbp_proj_dim,
            conv_filters=cbp_conv_filters,
            fusion_dim=fusion_dim,
            dropout=classifier_dropout,
        )

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(classifier_dropout),
            nn.Linear(fusion_dim // 2, 1),
        )

    def encode_image(self, pixel_values):
        image_map = self.image_encoder.features(pixel_values)
        image_map = F.relu(image_map, inplace=False)
        image_tokens = image_map.flatten(2).transpose(1, 2).contiguous()

        if self.use_image_attention_pooling:
            image_pooled, _ = self.image_attn_pool(image_tokens, attention_mask=None)
        else:
            image_pooled = image_tokens.mean(dim=1)

        image_features = self.image_projection(image_pooled)
        return image_features

    def forward(self, pixel_values, input_ids, attention_mask):
        image_features = self.encode_image(pixel_values)
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        if self.use_text_attention_pooling:
            text_features, _ = self.text_attn_pool(text_out.last_hidden_state, attention_mask=attention_mask)
        else:
            text_features = text_out.last_hidden_state[:, 0, :]

        fused = self.cbp(text_features, image_features)
        logits = self.classifier(fused).squeeze(-1)
        return logits


In [ ]:
# Define backbone freezing, selective unfreezing and parameter-count utilities.
def freeze_all_backbone_params(model):
    for param in model.image_encoder.parameters():
        param.requires_grad = False
    for param in model.text_encoder.parameters():
        param.requires_grad = False


def unfreeze_top_layers(
    model,
    num_densenet_blocks=NUM_UNFROZEN_DENSENET_BLOCKS,
    num_bert_layers=NUM_UNFROZEN_BERT_LAYERS,
):


    f = model.image_encoder.features
    densenet_blocks = [
        (f.denseblock1, f.transition1),
        (f.denseblock2, f.transition2),
        (f.denseblock3, f.transition3),
        (f.denseblock4, f.norm5),
    ]
    num_densenet_blocks = min(num_densenet_blocks, len(densenet_blocks))
    for block_pair in densenet_blocks[-num_densenet_blocks:]:
        for module in block_pair:
            for param in module.parameters():
                param.requires_grad = True

    bert_layers = model.text_encoder.encoder.layer
    for layer in bert_layers[-num_bert_layers:]:
        for param in layer.parameters():
            param.requires_grad = True

    if getattr(model.text_encoder, "pooler", None) is not None:
        for param in model.text_encoder.pooler.parameters():
            param.requires_grad = True


def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


In [ ]:
# Initialize the model, freeze both backbones and verify the image encoder weights.
model = MultimodalCBPClassifier().to(device)
freeze_all_backbone_params(model)

trainable, total = count_trainable_params(model)
print(f"Phase 1 trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

report = model.chexpert_load_report
assert report["missing_backbone_feature_keys"] == 0
assert report["unexpected_feature_keys"] == 0
assert report["differs_from_fresh_init"]

with torch.no_grad():
    sanity_batch = next(iter(train_loader))
    sanity_pixels = sanity_batch["pixel_values"].to(device)
    fmap = F.relu(model.image_encoder.features(sanity_pixels), inplace=False)
    tokens = fmap.flatten(2).transpose(1, 2).contiguous()
    image_embedding = model.encode_image(sanity_pixels)

print("DenseNet feature map shape :", tuple(fmap.shape))
print("DenseNet spatial tokens    :", tuple(tokens.shape))
print("Image embedding shape      :", tuple(image_embedding.shape))
print("CBP image/text input dims  :", (IMAGE_DIM, TEXT_DIM))

del sanity_batch, sanity_pixels, fmap, tokens, image_embedding
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 9. Class Balanced Focal Loss


In [ ]:
# Define focal loss for binary abnormality classification.
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_term = (1 - p_t).clamp(min=1e-6) ** self.gamma

        if self.alpha is not None:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            loss = alpha_t * focal_term * bce_loss
        else:
            loss = focal_term * bce_loss

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


In [ ]:
# Set focal loss weighting from the training set class distribution.

FOCAL_ALPHA = float(1.0 - pos_frac)
FOCAL_GAMMA = 2.0

criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
print(f"Using Focal Loss (alpha={FOCAL_ALPHA:.4f}, gamma={FOCAL_GAMMA}) "
      f"[positive-class prevalence: {pos_frac:.4f}]")


## 10. Memory-Efficient Training


In [ ]:
# Initialize automatic mixed precision gradient scaling.
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

## 11. Training and Evaluation Utilities


In [ ]:
# Define device transfer and metric computation utilities.
def move_batch_to_device(batch, device):
    return {
        "pixel_values": batch["pixel_values"].to(device, non_blocking=True),
        "input_ids": batch["input_ids"].to(device, non_blocking=True),
        "attention_mask": batch["attention_mask"].to(device, non_blocking=True),
        "label": batch["label"].to(device, non_blocking=True),
    }


In [ ]:
# Define one training epoch with mixed precision and gradient accumulation.
def train_one_epoch(model, loader, optimizer, criterion, scaler, scheduler=None, accum_steps=GRAD_ACCUM_STEPS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    progress = tqdm(loader, desc="Training", leave=False)

    for step, batch in enumerate(progress):
        batch = move_batch_to_device(batch, device)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(
                pixel_values=batch["pixel_values"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )
            loss = criterion(logits, batch["label"])
            loss = loss / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], max_norm=MAX_GRAD_NORM,
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()

        running_loss += loss.item() * accum_steps
        progress.set_postfix(loss=running_loss / (step + 1))

    return running_loss / len(loader)


In [ ]:
# Define validation evaluation and classification metrics.
@torch.no_grad()
def evaluate(model, loader, criterion, threshold=0.5):
    model.eval()
    running_loss = 0.0

    all_labels = []
    all_probs = []

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        batch = move_batch_to_device(batch, device)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(
                pixel_values=batch["pixel_values"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )
            loss = criterion(logits, batch["label"])

        running_loss += loss.item()

        probs = torch.sigmoid(logits).float().cpu().numpy()
        labels = batch["label"].cpu().numpy()

        all_probs.append(probs)
        all_labels.append(labels)

    all_probs = np.concatenate(all_probs).ravel()
    all_labels = np.concatenate(all_labels).ravel()
    preds = (all_probs >= threshold).astype(int)

    metrics = {
        "loss": running_loss / len(loader),
        "accuracy": accuracy_score(all_labels, preds),
        "precision": precision_score(all_labels, preds, zero_division=0),
        "recall": recall_score(all_labels, preds, zero_division=0),
        "f1": f1_score(all_labels, preds, zero_division=0),
        "auroc": roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else float("nan"),
    }

    return metrics, all_labels, all_probs


## 12. Phase 1 : Fusion-Head Warmup


In [ ]:
# Train the fusion head while both pretrained backbones remain frozen.
def build_head_only_optimizer(model, head_lr=HEAD_LR, weight_decay=WEIGHT_DECAY):
    head_params = [p for n, p in model.named_parameters() if p.requires_grad]
    print(f"Phase 1 trainable (head-only) params: {sum(p.numel() for p in head_params):,}")
    return torch.optim.AdamW(head_params, lr=head_lr, weight_decay=weight_decay)


history = {"phase": [], "train_loss": [], "val_loss": [], "val_auroc": [], "val_f1": []}

phase1_optimizer = build_head_only_optimizer(model)

best_val_auroc = -np.inf
epochs_without_improvement = 0

for epoch in range(1, PHASE1_EPOCHS + 1):
    print(f"\n=== Phase 1 (head warmup) — Epoch {epoch}/{PHASE1_EPOCHS} ===")

    train_loss = train_one_epoch(model, train_loader, phase1_optimizer, criterion, scaler)
    val_metrics, _, _ = evaluate(model, val_loader, criterion)

    history["phase"].append(1)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_auroc"].append(val_metrics["auroc"])
    history["val_f1"].append(val_metrics["f1"])

    print(
        f"Train loss: {train_loss:.4f} | Val loss: {val_metrics['loss']:.4f} | "
        f"Val AUROC: {val_metrics['auroc']:.4f} | Val F1: {val_metrics['f1']:.4f}"
    )

    if val_metrics["auroc"] > best_val_auroc:
        best_val_auroc = val_metrics["auroc"]
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  -> New best model saved (Val AUROC: {best_val_auroc:.4f})")

print(f"\nPhase 1 complete. Best validation AUROC so far: {best_val_auroc:.4f}")


## 13. Phase 2 : End to End Fine Tuning




In [ ]:
# Unfreeze the selected upper backbone layers for fine-tuning.
unfreeze_top_layers(model)

trainable, total = count_trainable_params(model)
print(f"Phase 2 trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")


In [ ]:
# Build differential optimizer groups and the Phase-2 learning rate scheduler.
def build_param_groups(model, backbone_lr=BACKBONE_LR, head_lr=HEAD_LR, weight_decay=WEIGHT_DECAY):
    backbone_params = [
        p for n, p in model.named_parameters()
        if p.requires_grad and (n.startswith("image_encoder") or n.startswith("text_encoder"))
    ]
    head_params = [
        p for n, p in model.named_parameters()
        if p.requires_grad and not (n.startswith("image_encoder") or n.startswith("text_encoder"))
    ]

    print(f"Backbone trainable params: {sum(p.numel() for p in backbone_params):,}")
    print(f"Head trainable params: {sum(p.numel() for p in head_params):,}")

    return [
        {"params": backbone_params, "lr": backbone_lr, "weight_decay": weight_decay},
        {"params": head_params, "lr": head_lr, "weight_decay": weight_decay},
    ]


optimizer = torch.optim.AdamW(build_param_groups(model))

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps_phase2 = steps_per_epoch * PHASE2_MAX_EPOCHS
warmup_steps_phase2 = max(1, int(WARMUP_RATIO_PHASE2 * total_steps_phase2))

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps_phase2,
    num_training_steps=total_steps_phase2,
)

print(f"Phase 2: {steps_per_epoch} optimizer steps/epoch, "
      f"{total_steps_phase2} total steps, {warmup_steps_phase2} warmup steps")


In [ ]:
# Fine-tune the model and save the checkpoint with the best validation AUROC.

epochs_without_improvement = 0

for epoch in range(1, PHASE2_MAX_EPOCHS + 1):
    print(f"\n=== Phase 2 (end-to-end fine-tuning) — Epoch {epoch}/{PHASE2_MAX_EPOCHS} ===")

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, scheduler=scheduler)
    val_metrics, _, _ = evaluate(model, val_loader, criterion)

    history["phase"].append(2)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_auroc"].append(val_metrics["auroc"])
    history["val_f1"].append(val_metrics["f1"])

    current_lr_backbone = optimizer.param_groups[0]["lr"]
    current_lr_head = optimizer.param_groups[1]["lr"]

    print(
        f"Train loss: {train_loss:.4f} | Val loss: {val_metrics['loss']:.4f} | "
        f"Val AUROC: {val_metrics['auroc']:.4f} | Val F1: {val_metrics['f1']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"LR (backbone/head): {current_lr_backbone:.2e}/{current_lr_head:.2e}"
    )

    if val_metrics["auroc"] > best_val_auroc:
        best_val_auroc = val_metrics["auroc"]
        epochs_without_improvement = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  -> New best model saved (Val AUROC: {best_val_auroc:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  -> No improvement for {epochs_without_improvement} epoch(s)")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping triggered after {epoch} phase-2 epochs.")
        break

print(f"\nBest validation AUROC overall: {best_val_auroc:.4f}")
print(f"Best model saved to: {BEST_MODEL_PATH}")


## 14. Best Checkpoint Restoration


In [ ]:
# Restore the best validation selected model checkpoint for final evaluation.
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()
print("Loaded best checkpoint from:", BEST_MODEL_PATH)


## 15. Validation-Based Decision Threshold Selection


In [ ]:
# Select the decision threshold that maximizes validation F1 score.
_, val_labels, val_probs = evaluate(model, val_loader, criterion, threshold=0.5)

candidate_thresholds = np.linspace(0.05, 0.95, 181)
best_threshold = 0.5
best_f1 = -1.0

for t in candidate_thresholds:
    preds = (val_probs >= t).astype(int)
    f1 = f1_score(val_labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"Best validation F1: {best_f1:.4f} at threshold = {best_threshold:.3f}")

In [ ]:
# Report validation metrics at the selected decision threshold.

val_preds = (val_probs >= best_threshold).astype(int)
val_accuracy = accuracy_score(val_labels, val_preds)
val_precision = precision_score(
    val_labels,
    val_preds,
    zero_division=0
)
val_recall = recall_score(
    val_labels,
    val_preds,
    zero_division=0
)
val_f1 = f1_score(
    val_labels,
    val_preds,
    zero_division=0
)
val_mcc = matthews_corrcoef(
    val_labels,
    val_preds
)
val_auroc = roc_auc_score(
    val_labels,
    val_probs
)

val_cm = confusion_matrix(
    val_labels,
    val_preds
)

tn, fp, fn, tp = val_cm.ravel()

val_specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
val_sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
val_npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0

validation_results = {
    "Threshold": best_threshold,
    "Accuracy": val_accuracy,
    "Precision": val_precision,
    "Recall / Sensitivity": val_recall,
    "Specificity": val_specificity,
    "F1": val_f1,
    "MCC": val_mcc,
    "AUROC": val_auroc,
    "NPV": val_npv,
    "TP": tp,
    "TN": tn,
    "FP": fp,
    "FN": fn,
}

validation_results_df = pd.DataFrame(
    [validation_results]
)

print("Validation Set Performance")
display(validation_results_df)

print("\nConfusion Matrix:")
print(val_cm)


## 16. Final Held-Out Test Evaluation


In [ ]:
# Evaluate the final model on the held-out test set and store predictions.
@torch.no_grad()
def evaluate_test(model, loader, threshold):
    model.eval()

    all_uids = []
    all_labels = []
    all_probs = []

    for batch in tqdm(loader, desc="Test evaluation", leave=False):
        uids = batch["uid"]
        moved = move_batch_to_device(batch, device)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(
                pixel_values=moved["pixel_values"],
                input_ids=moved["input_ids"],
                attention_mask=moved["attention_mask"],
            )

        probs = torch.sigmoid(logits).float().cpu().numpy()
        labels = moved["label"].cpu().numpy()

        all_uids.extend(uids)
        all_probs.append(probs)
        all_labels.append(labels)

    all_probs = np.concatenate(all_probs).ravel()
    all_labels = np.concatenate(all_labels).ravel()
    preds = (all_probs >= threshold).astype(int)

    results = {
        "Threshold": threshold,
        "Accuracy": accuracy_score(all_labels, preds),
        "Precision": precision_score(all_labels, preds, zero_division=0),
        "Recall": recall_score(all_labels, preds, zero_division=0),
        "F1": f1_score(all_labels, preds, zero_division=0),
        "MCC": matthews_corrcoef(all_labels, preds),
        "AUROC": roc_auc_score(all_labels, all_probs),
    }

    cm = confusion_matrix(all_labels, preds)

    predictions_df = pd.DataFrame({
        "uid": all_uids,
        "true_label": all_labels.astype(int),
        "predicted_label": preds,
        "prob_abnormal": all_probs,
    })
    predictions_df["distance_from_threshold"] = predictions_df["prob_abnormal"] - threshold

    return results, cm, predictions_df

test_results, test_confusion_matrix, test_predictions_df = evaluate_test(model, test_loader, best_threshold)

print("Final Test Set Results")
for k, v in test_results.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

print("\nConfusion Matrix:")
print(test_confusion_matrix)


In [ ]:
# Save the final held-out test metrics.
results_df = pd.DataFrame([test_results])
results_path = os.path.join(MODEL_FOLDER, "test_results_cbp_e2e_v2.csv")
results_df.to_csv(results_path, index=False)
print("Saved test results to:", results_path)
results_df


# Part II — Modality Contribution Analysis

In [ ]:
# Snapshot the trained model state before modality contribution analysis.
import copy

state_before = {
    k: v.detach().cpu().clone()
    for k, v in model.state_dict().items()
}


In [ ]:
# Define inference with image or text features removed after projection.

@torch.no_grad()
def forward_with_modality_ablation(
    model,
    pixel_values,
    input_ids,
    attention_mask,
    condition="multimodal"
):
    """
    condition:
        "multimodal" -> Image + Text
        "text_only"  -> image contribution removed
        "image_only" -> text contribution removed

    Ablation is performed AFTER the CBP projection layers.
    The trained model and its parameters remain unchanged.
    """

    model.eval()

    image_map = model.image_encoder.features(pixel_values)
    image_map = F.relu(image_map, inplace=False)
    image_tokens = image_map.flatten(2).transpose(1, 2).contiguous()

    if model.use_image_attention_pooling:
        image_pooled, _ = model.image_attn_pool(
            image_tokens,
            attention_mask=None
        )
    else:
        image_pooled = image_tokens.mean(dim=1)

    image_features = model.image_projection(image_pooled)

    text_out = model.text_encoder(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    if model.use_text_attention_pooling:
        text_features, _ = model.text_attn_pool(
            text_out.last_hidden_state,
            attention_mask=attention_mask
        )
    else:
        text_features = text_out.last_hidden_state[:, 0, :]

    t = model.cbp.proj_dropout(
        model.cbp.text_proj(text_features)
    )

    v = model.cbp.proj_dropout(
        model.cbp.image_proj(image_features)
    )

    if condition == "text_only":
        v = torch.zeros_like(v)

    elif condition == "image_only":
        t = torch.zeros_like(t)

    elif condition == "multimodal":
        pass

    else:
        raise ValueError(
            "condition must be 'multimodal', "
            "'text_only', or 'image_only'"
        )

    bilinear_map = torch.bmm(
        v.unsqueeze(2),
        t.unsqueeze(1)
    )

    bilinear_map = bilinear_map.unsqueeze(1)
    compressed = model.cbp.conv_block(
        bilinear_map
    )
    compressed = compressed.flatten(
        start_dim=1
    )
    fusion_input = torch.cat(
        [compressed, t, v],
        dim=1
    )

    fused = model.cbp.fusion_proj(
        fusion_input
    )
    logits = model.classifier(
        fused
    ).squeeze(-1)

    return logits


In [ ]:
# Evaluate the fixed trained model under each modality condition.

@torch.no_grad()
def evaluate_modality_condition(
    model,
    loader,
    threshold,
    condition
):

    model.eval()
    all_labels = []
    all_probs = []

    for batch in tqdm(
        loader,
        desc=f"Evaluating {condition}",
        leave=False
    ):

        moved = move_batch_to_device(
            batch,
            device
        )

        with torch.amp.autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):

            logits = forward_with_modality_ablation(
                model=model,
                pixel_values=moved["pixel_values"],
                input_ids=moved["input_ids"],
                attention_mask=moved["attention_mask"],
                condition=condition
            )

        probs = torch.sigmoid(
            logits
        ).float().cpu().numpy()

        labels = moved[
            "label"
        ].cpu().numpy()

        all_probs.append(probs)
        all_labels.append(labels)

    all_probs = np.concatenate(
        all_probs
    ).ravel()

    all_labels = np.concatenate(
        all_labels
    ).ravel()

    preds = (
        all_probs >= threshold
    ).astype(int)

    results = {
        "Condition": condition,

        "Accuracy": accuracy_score(
            all_labels,
            preds
        ),

        "Precision": precision_score(
            all_labels,
            preds,
            zero_division=0
        ),

        "Recall": recall_score(
            all_labels,
            preds,
            zero_division=0
        ),

        "F1": f1_score(
            all_labels,
            preds,
            zero_division=0
        ),

        "MCC": matthews_corrcoef(
            all_labels,
            preds
        ),

        "AUROC": roc_auc_score(
            all_labels,
            all_probs
        )
    }

    return results


In [ ]:
# Run multimodal, text-only, and image-only evaluations on the test set.

multimodal_results = evaluate_modality_condition(
    model,
    test_loader,
    best_threshold,
    condition="multimodal"
)

text_only_results = evaluate_modality_condition(
    model,
    test_loader,
    best_threshold,
    condition="text_only"
)

image_only_results = evaluate_modality_condition(
    model,
    test_loader,
    best_threshold,
    condition="image_only"
)

modality_results_df = pd.DataFrame([
    multimodal_results,
    text_only_results,
    image_only_results
])

modality_results_df["Condition"] = [
    "Image + Text",
    "Text only",
    "Image only"
]

display(modality_results_df)


## 17. Unimodal Ablation Drop


In [ ]:
# Compute AUROC drops caused by removing each modality.

M_full = multimodal_results["AUROC"]
M_text = text_only_results["AUROC"]
M_image = image_only_results["AUROC"]

image_ablation_drop = M_full - M_text
text_ablation_drop = M_full - M_image

print("=== AUROC Ablation Drop ===")

print(f"Multimodal AUROC : {M_full:.4f}")
print(f"Text-only AUROC  : {M_text:.4f}")
print(f"Image-only AUROC : {M_image:.4f}")
print()
print(f"Image ablation drop : "
    f"{image_ablation_drop:.4f}"
)
print(
    f"Text ablation drop  : "
    f"{text_ablation_drop:.4f}"
)

## 18. Modality Attribution Ratio


In [ ]:
# Convert positive AUROC drops into relative modality attribution percentages.

total_drop = (
    image_ablation_drop
    +
    text_ablation_drop
)
if (
    image_ablation_drop >= 0
    and text_ablation_drop >= 0
    and total_drop > 0
):
    image_attribution = (
        image_ablation_drop
        /
        total_drop
        *
        100
    )

    text_attribution = (
        text_ablation_drop
        /
        total_drop
        *
        100
    )

    print(
        f"Image attribution : "
        f"{image_attribution:.2f}%"
    )
    print(
        f"Text attribution  : "
        f"{text_attribution:.2f}%"
    )

else:
    print(
        "Attribution ratio not computed because "
        "one modality ablation did not reduce AUROC."
    )

In [ ]:
# Verify that modality analysis did not modify the trained model weights.
unchanged = all(
    torch.equal(
        state_before[k],
        v.detach().cpu()
    )
    for k, v in model.state_dict().items()
)

print("Model weights unchanged:", unchanged)